# Unit 4 Assignment: Evaluated Agentic RAG System

This notebook implements a **self-evaluating agentic RAG system** using:
- **CrewAI** for multi-agent orchestration
- **LangChain + FAISS** for retrieval-augmented generation
- **DeepEval** for answer quality evaluation
- **Groq** as the LLM backend

**Topic chosen**: The James Webb Space Telescope (JWST) — a rich scientific topic with many distinct, verifiable facts, making it ideal for testing RAG faithfulness and relevancy.

## 0. Installation & Setup

In [1]:
!pip install -q crewai crewai-tools langchain langchain-community langchain-groq \
    faiss-cpu sentence-transformers deepeval groq python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.2/784.2 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.4/843.4 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10

In [9]:
!pip install -q langchain-text-splitters

In [19]:
!pip install -q litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai-tools 1.14.2 requires tiktoken~=0.8.0, but you have tiktoken 0.12.0 which is incompatible.
deepeval 3.9.6 requi

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Use Groq as OpenAI-compatible backend for DeepEval
os.environ["OPENAI_API_KEY"] = "" #removed hardcoded key
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

# Keep Groq key available separately too
os.environ["GROQ_API_KEY"] = ""

print(f"GROQ_API_KEY: {'set' if GROQ_API_KEY else 'NOT SET — add to .env'}")
print(f"OPENAI_API_KEY mapped to Groq: {'set' if GROQ_API_KEY else 'NOT SET'}")

GROQ_API_KEY: NOT SET — add to .env
OPENAI_API_KEY mapped to Groq: NOT SET


---
## Part 1: Knowledge Base (10 marks)

**Topic**: The James Webb Space Telescope (JWST)

**Why chosen**: JWST is a rich scientific topic with dozens of precise, verifiable facts (mirror diameter, launch date, orbit, instruments, discoveries). This makes it perfect for testing *faithfulness* (does the answer stick to retrieved facts?) and *relevancy* (does the answer address the question?). Adversarial questions (e.g., about older telescopes not in the corpus) are easy to design, letting us test graceful degradation.

In [2]:
# ── Knowledge Base Text ───────────────────────────────────────────────────────
JWST_TEXT = """
The James Webb Space Telescope (JWST) is a large, space-based observatory designed to conduct
infrared astronomy. It is the most powerful space telescope ever built and is a successor to the
Hubble Space Telescope. JWST was launched on December 25, 2021, aboard an Ariane 5 rocket from
the Guiana Space Centre in Kourou, French Guiana.

JWST is a collaborative project between NASA, the European Space Agency (ESA), and the Canadian
Space Agency (CSA). The telescope was named after James E. Webb, who served as NASA Administrator
from 1961 to 1968 and oversaw the Apollo program.

The telescope's primary mirror is 6.5 meters (21.3 feet) in diameter and is composed of 18
hexagonal gold-plated beryllium mirror segments. This is significantly larger than Hubble's
2.4-meter mirror, giving JWST roughly 6.25 times the light-collecting area of Hubble.

JWST operates at the second Sun-Earth Lagrange point (L2), approximately 1.5 million kilometers
(about 1 million miles) from Earth. This location allows the telescope to maintain a stable
thermal environment and keep its sunshield between itself and the Sun, Earth, and Moon simultaneously.

The five-layer sunshield is roughly the size of a tennis court (21 meters by 14 meters). It blocks
solar radiation and keeps the telescope's instruments cold enough to detect infrared light. The
sunshield maintains a temperature difference of about 300 degrees Celsius between its hot and cold
sides, keeping the cold side below -233 degrees Celsius (40 Kelvin).

JWST carries four main science instruments:
1. NIRCam (Near Infrared Camera) - the primary imager, covering wavelengths from 0.6 to 5 microns.
2. NIRSpec (Near Infrared Spectrograph) - can observe up to 100 objects simultaneously.
3. MIRI (Mid-Infrared Instrument) - covers mid-infrared wavelengths from 5 to 28 microns,
   jointly developed by ESA and NASA.
4. FGS/NIRISS (Fine Guidance Sensor / Near Infrared Imager and Slitless Spectrograph) -
   contributed by the Canadian Space Agency.

In July 2022, NASA released the first full-color science images from JWST. These included the
deepest infrared image of the universe ever taken (showing galaxy cluster SMACS 0723), the
Carina Nebula, Stephan's Quintet, the Southern Ring Nebula, and the first direct detection of
carbon dioxide in a exoplanet atmosphere (WASP-39b).

One of JWST's primary science goals is to observe the first galaxies that formed in the early
universe, shortly after the Big Bang. It can see light from galaxies formed as early as 100-250
million years after the Big Bang, a period referred to as Cosmic Dawn.

JWST discovered a galaxy, JADES-GS-z14-0, that formed just 290 million years after the Big Bang,
making it the oldest galaxy ever observed. The galaxy showed unexpected brightness and had signs
of oxygen, which was surprising given how early in cosmic history it formed.

JWST has also made significant contributions to exoplanet science. Beyond the WASP-39b CO2
detection, it detected methane and carbon dioxide in the atmosphere of K2-18b, a sub-Neptune
exoplanet that orbits in the habitable zone of its star. These molecules could be consistent with
a water ocean under a hydrogen-rich atmosphere (a 'Hycean' world).

The telescope has a design lifetime of at least 10 years. However, due to the precision of the
Ariane 5 launch, JWST used less propellant than planned for orbit insertion, potentially extending
its operational lifetime to 20 years or more.

JWST images are primarily captured in infrared wavelengths, which are invisible to human eyes.
Scientists translate these infrared data into visible-light images by mapping different infrared
wavelengths to visible colors — a process called false-color mapping.

The total cost of JWST was approximately $10 billion USD, making it one of the most expensive
scientific instruments ever built. Development took over 20 years and involved contributions from
thousands of scientists and engineers across 17 countries.

JWST studied the atmosphere of TRAPPIST-1c, a rocky planet in the TRAPPIST-1 system. Results
suggested the planet has little to no atmosphere, or possibly a thin carbon dioxide atmosphere
with no significant water vapor — important for understanding habitability in that system.

The telescope's NIRCam instrument discovered Earendel, a single star nicknamed the 'Sunrise Arc'
star, located 12.9 billion light-years away — the most distant individual star ever observed.
This was initially found by Hubble but JWST confirmed and expanded observations.

JWST is contributing to Solar System science as well. It has imaged Jupiter in remarkable detail
showing never-before-seen features in its atmosphere, auroras, and rings. It has also studied
Neptune's rings and moons with unprecedented clarity.
"""

print(f'Knowledge base loaded: {len(JWST_TEXT.split())} words, {len(JWST_TEXT)} characters')

Knowledge base loaded: 732 words, 4773 characters


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# __ Split text into chunks ____________________________________________________
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=['\n\n', '\n', '. ', ' ']
)
docs = splitter.create_documents([JWST_TEXT])
print(f'Created {len(docs)} chunks from the knowledge base.')
for i, d in enumerate(docs):
    print(f'  Chunk {i+1}: {len(d.page_content)} chars')

# __ Build FAISS vector store with HuggingFace embeddings _____________________
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)
vector_store = FAISS.from_documents(docs, embeddings)
print('\nFAISS vector store built successfully.')

# __ Quick smoke test __________________________________________________________
test_results = vector_store.similarity_search('How large is JWST mirror?', k=2)
print('\nSmoke test retrieval for "How large is JWST mirror?"_')
for r in test_results:
    print(f'  -> {r.page_content[:120]}...')

Created 17 chunks from the knowledge base.
  Chunk 1: 335 chars
  Chunk 2: 243 chars
  Chunk 3: 268 chars
  Chunk 4: 290 chars
  Chunk 5: 362 chars
  Chunk 6: 358 chars
  Chunk 7: 167 chars
  Chunk 8: 331 chars
  Chunk 9: 260 chars
  Chunk 10: 270 chars
  Chunk 11: 348 chars
  Chunk 12: 239 chars
  Chunk 13: 261 chars
  Chunk 14: 250 chars
  Chunk 15: 278 chars
  Chunk 16: 271 chars
  Chunk 17: 244 chars


/tmp/ipykernel_16988/2119674219.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



FAISS vector store built successfully.

Smoke test retrieval for "How large is JWST mirror?"_
  -> The telescope's primary mirror is 6.5 meters (21.3 feet) in diameter and is composed of 18
hexagonal gold-plated berylli...
  -> The James Webb Space Telescope (JWST) is a large, space-based observatory designed to conduct
infrared astronomy. It is ...


---
## Part 2: RAG Agent (20 marks)

We define a CrewAI `Agent` with a `@tool`-decorated retrieval function. The task output includes **both the generated answer and the retrieved context** so the evaluator agent can measure faithfulness.

In [19]:
from crewai import Agent, Task, Crew
from crewai.tools import tool
import os

llm = "groq/llama-3.1-8b-instant"

@tool("JWST Knowledge Base Search")
def jwst_search_tool(query: str) -> str:
    """
    Search the JWST vector database and return the most relevant passages.
    Input: natural language query about James Webb Space Telescope.
    Output: relevant context passages.
    """

    results = vector_store.similarity_search(query, k=4)

    if not results:
        return "No relevant information found."

    return "\n\n".join(
        [f"[Passage {i+1}] {doc.page_content}" for i, doc in enumerate(results)]
    )

rag_agent = Agent(
    role="JWST RAG Retriever",
    goal="Answer JWST questions using retrieved context.",
    backstory="Expert in JWST facts.",
    tools=[jwst_search_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("RAG Agent ready")

RAG Agent ready


In [5]:
def make_rag_task(question: str) -> Task:
    """Factory that creates a RAG task for a given question."""
    return Task(
        description=(
            f'Answer the following question using the JWST knowledge base:\n\n'
            f'QUESTION: {question}\n\n'
            f'Steps:\n'
            f'1. Use the JWST Knowledge Base Search tool to retrieve relevant passages.\n'
            f'2. Formulate a clear, concise answer grounded ONLY in the retrieved passages.\n'
            f'3. Return your output in EXACTLY this format:\n'
            f'   ANSWER: <your answer here>\n'
            f'   CONTEXT: <paste the full retrieved passages here>'
        ),
        expected_output=(
            'A structured response with two clearly labelled sections:\n'
            'ANSWER: <concise answer to the question>\n'
            'CONTEXT: <the retrieved passages used to generate the answer>'
        ),
        agent=rag_agent
    )

print('RAG task factory defined.')

RAG task factory defined.


In [7]:
from crewai import Process

# ── Sample outputs for 3 test questions ──────────────────────────────────────
TEST_QUESTIONS_SAMPLE = [
    'What is the diameter of the JWST primary mirror?',
    'What are the four main science instruments on JWST?',
    'Where does JWST orbit and why was that location chosen?'
]

sample_outputs = {}

for q in TEST_QUESTIONS_SAMPLE:
    print(f'\n{"="*60}')
    print(f'QUESTION: {q}')
    print('='*60)
    rag_task = make_rag_task(q)
    crew = Crew(agents=[rag_agent], tasks=[rag_task], process=Process.sequential, verbose=False)
    result = crew.kickoff()
    sample_outputs[q] = str(result)
    print(result)


QUESTION: What is the diameter of the JWST primary mirror?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: JWST RAG Retriever                                                                                      │
│                                                                                                                 │
│  Task: Answer the following question using the JWST knowledge base:                                             │
│                                                                                                                 │
│  QUESTION: What is the diameter of the JWST primary mirror?                                                     │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the JWST Knowledge Base Search tool to retrieve relevant passages.                                      │
│  2. Formulate a clear, concise answer grounded ONLY in the retrieved passages.                                  │
│  3. Return your output in EXACTLY this format:                                                                  │
│     ANSWER: <your answer here>                                                                                  │
│     CONTEXT: <paste the full retrieved passages here>                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool jwst_knowledge_base_search executed with result: [Passage 1] The telescope's primary mirror is 6.5 meters (21.3 feet) in diameter and is composed of 18
hexagonal gold-plated beryllium mirror segments. This is significantly larger than Hubble's
2.4-m...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: JWST RAG Retriever                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ANSWER: 6.5 meters                                                                                             │
│  CONTEXT: [Passage 1] The telescope's primary mirror is 6.5 meters (21.3 feet) in diameter and is composed of   │
│  18                                                                                                             │
│  hexagonal gold-plated beryllium mirror segments. This is significantly larger than Hubble's                    │
│  2.4-meter mirror, giving JWST roughly 6.25 times the light-collecting area of Hubble.                          │
│                                                                                                                 │
│  [Passage 2] JWST operates at the second Sun-Earth Lagrange point (L2), approximately 1.5 million kilometers    │
│  (about 1 million miles) from Earth. This location allows the telescope to maintain a stable                    │
│  thermal environment and keep its sunshield between itself and the Sun, Earth, and Moon simultaneously.         │
│                                                                                                                 │
│  [Passage 3] The James Webb Space Telescope (JWST) is a large, space-based observatory designed to conduct      │
│  infrared astronomy. It is the most powerful space telescope ever built and is a successor to the               │
│  Hubble Space Telescope. JWST was launched on December 25, 2021, aboard an Ariane 5 rocket from                 │
│  the Guiana Space Centre in Kourou, French Guiana.                                                              │
│                                                                                                                 │
│  [Passage 4] JWST is a collaborative project between NASA, the European Space Agency (ESA), and the Canadian    │
│  Space Agency (CSA). The telescope was named after James E. Webb, who served as NASA Administrator              │
│  from 1961 to 1968 and oversaw the Apollo program.                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: JWST RAG Retriever                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ANSWER: JWST orbits at the second Sun-Earth Lagrange point (L2), approximately 1.5 million kilometers from     │
│  Earth, to maintain a stable thermal environment and keep its sunshield between itself and the Sun, Earth, and  │
│  Moon.                                                                                                          │
│  CONTEXT: [Passage 1] JWST operates at the second Sun-Earth Lagrange point (L2), approximately 1.5 million      │
│  kilometers                                                                                                     │
│  (about 1 million miles) from Earth. This location allows the telescope to maintain a stable                    │
│  thermal environment and keep its sunshield between itself and the Sun, Earth, and Moon simultaneously.         │
│                                                                                                                 │
│  [Passage 2] The James Webb Space Telescope (JWST) is a large, space-based observatory designed to conduct      │
│  infrared astronomy. It is the most powerful space telescope ever built and is a successor to the               │
│  Hubble Space Telescope. JWST was launched on December 25, 2021, aboard an Ariane 5 rocket from                 │
│  the Guiana Space Centre in Kourou, French Guiana.                                                              │
│                                                                                                                 │
│  [Passage 3] JWST is a collaborative project between NASA, the European Space Agency (ESA), and the Canadian    │
│  Space Agency (CSA). The telescope was named after James E. Webb, who served as NASA Administrator              │
│  from 1961 to 1968 and oversaw the Apollo program.                                                              │
│                                                                                                                 │
│  [Passage 4] The telescope has a design lifetime of at least 10 years. However, due to the precision of the     │
│  Ariane 5 launch, JWST used less propellant than planned for orbit insertion, potentially extending             │
│  its operational lifetime to 20 years or more.                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

ANSWER: JWST orbits at the second Sun-Earth Lagrange point (L2), approximately 1.5 million kilometers from Earth, to maintain a stable thermal environment and keep its sunshield between itself and the Sun, Earth, and Moon.
CONTEXT: [Passage 1] JWST operates at the second Sun-Earth Lagrange point (L2), approximately 1.5 million kilometers 
(about 1 million miles) from Earth. This location allows the telescope to maintain a stable 
thermal environment and keep its sunshield between itself and the Sun, Earth, and Moon simultaneously.

[Passage 2] The James Webb Space Telescope (JWST) is a large, space-based observatory designed to conduct 
infrared astronomy. It is the most powerful space telescope ever built and is a successor to the 
Hubble Space Telescope. JWST was launched on December 25, 2021, aboard an Ariane 5 rocket from 
the Guiana Space Centre in Kourou, French Guiana.

[Passage 3] JWST is a collaborative project between NASA, the European Space Agency (ESA), and the Canadian 
S

---
## Part 3: Quality Evaluator Agent (25 marks)

The evaluator agent wraps **DeepEval's FaithfulnessMetric and AnswerRelevancyMetric** inside a `@tool`. It takes the RAG output, parses the answer and context, runs both metrics, and returns a structured verdict with scores, pass/fail status, and specific failure reasons.

In [8]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

EVAL_THRESHOLD = 0.7

# ── Evaluation Tool ───────────────────────────────────────────────────────────
@tool('DeepEval Quality Evaluator')
def evaluate_rag_output(rag_output: str, question: str) -> str:
    """Evaluate a RAG answer for faithfulness and relevancy using DeepEval.

    Args:
        rag_output: The full output from the RAG agent, containing ANSWER: and CONTEXT: sections.
        question: The original user question.

    Returns:
        A JSON string with faithfulness_score, relevancy_score, verdict, and reasons.
    """
    # ── Parse ANSWER and CONTEXT from RAG output ──────────────────────────────
    answer = ''
    context_text = ''

    if 'ANSWER:' in rag_output and 'CONTEXT:' in rag_output:
        parts = rag_output.split('CONTEXT:')
        answer_part = parts[0].replace('ANSWER:', '').strip()
        context_text = parts[1].strip() if len(parts) > 1 else ''
        answer = answer_part
    else:
        # Fallback: treat whole output as answer
        answer = rag_output.strip()
        context_text = rag_output.strip()

    if not answer:
        return json.dumps({
            'faithfulness_score': 0.0,
            'relevancy_score': 0.0,
            'verdict': 'FAIL',
            'reasons': ['Could not parse answer from RAG output.']
        })

    # ── Build DeepEval test case ───────────────────────────────────────────────
    retrieval_context = [ctx.strip() for ctx in context_text.split('[Passage') if ctx.strip()]
    if not retrieval_context:
        retrieval_context = [context_text]

    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=retrieval_context
    )

    # ── Run metrics ───────────────────────────────────────────────────────────
    results = {'faithfulness_score': 0.0, 'relevancy_score': 0.0, 'verdict': 'FAIL', 'reasons': []}

    try:
        faith_metric = FaithfulnessMetric(threshold=EVAL_THRESHOLD, verbose_mode=False)
        faith_metric.measure(test_case)
        results['faithfulness_score'] = round(faith_metric.score, 3)
        if faith_metric.score < EVAL_THRESHOLD:
            results['reasons'].append(
                f'Faithfulness FAIL ({faith_metric.score:.2f} < {EVAL_THRESHOLD}): '
                f'{faith_metric.reason}'
            )
    except Exception as e:
        results['reasons'].append(f'Faithfulness metric error: {str(e)}')

    try:
        rel_metric = AnswerRelevancyMetric(threshold=EVAL_THRESHOLD, verbose_mode=False)
        rel_metric.measure(test_case)
        results['relevancy_score'] = round(rel_metric.score, 3)
        if rel_metric.score < EVAL_THRESHOLD:
            results['reasons'].append(
                f'Relevancy FAIL ({rel_metric.score:.2f} < {EVAL_THRESHOLD}): '
                f'{rel_metric.reason}'
            )
    except Exception as e:
        results['reasons'].append(f'Relevancy metric error: {str(e)}')

    # ── Determine overall verdict ─────────────────────────────────────────────
    both_pass = (
        results['faithfulness_score'] >= EVAL_THRESHOLD and
        results['relevancy_score'] >= EVAL_THRESHOLD
    )
    results['verdict'] = 'PASS' if both_pass else 'FAIL'

    return json.dumps(results, indent=2)


# ── Evaluator Agent ───────────────────────────────────────────────────────────
evaluator_agent = Agent(
    role='Answer Quality Evaluator',
    goal=(
        'Evaluate the quality of RAG-generated answers using DeepEval metrics. '
        'Provide detailed scores, a PASS/FAIL verdict, and specific reasons for any failures.'
    ),
    backstory=(
        'You are a quality assurance specialist for AI systems. You rigorously evaluate '
        'answers for faithfulness to source documents and relevance to the user question. '
        'You always provide actionable, specific feedback when answers fail quality thresholds.'
    ),
    tools=[evaluate_rag_output],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print('Evaluator agent defined.')

Evaluator agent defined.


In [9]:
def make_eval_task(question: str, rag_task: Task) -> Task:
    """Factory that creates an evaluation task that depends on a RAG task."""
    return Task(
        description=(
            f'Evaluate the quality of the RAG agent\'s answer to this question:\n'
            f'ORIGINAL QUESTION: {question}\n\n'
            f'Steps:\n'
            f'1. Read the RAG agent\'s output from context (it contains ANSWER: and CONTEXT: sections).\n'
            f'2. Call the DeepEval Quality Evaluator tool with the full RAG output and the original question.\n'
            f'3. Report the evaluation results including:\n'
            f'   - Faithfulness score\n'
            f'   - Relevancy score\n'
            f'   - Overall verdict (PASS/FAIL)\n'
            f'   - Specific reasons for any failures\n'
        ),
        expected_output=(
            'A structured evaluation report containing:\n'
            'FAITHFULNESS_SCORE: <score between 0 and 1>\n'
            'RELEVANCY_SCORE: <score between 0 and 1>\n'
            'VERDICT: <PASS or FAIL>\n'
            'REASONS: <specific reasons for failures, or "None - all metrics passed">\n'
            'RAW_JSON: <the full JSON from the evaluation tool>'
        ),
        agent=evaluator_agent,
        context=[rag_task]
    )

print('Evaluator task factory defined.')

Evaluator task factory defined.


In [10]:
# ── Demo: evaluate one sample question ───────────────────────────────────────
demo_q = 'What is the diameter of the JWST primary mirror?'
demo_rag_task = make_rag_task(demo_q)
demo_eval_task = make_eval_task(demo_q, demo_rag_task)

demo_crew = Crew(
    agents=[rag_agent, evaluator_agent],
    tasks=[demo_rag_task, demo_eval_task],
    process=Process.sequential,
    verbose=True
)

demo_result = demo_crew.kickoff()
print('\n' + '='*60)
print('EVALUATION RESULT:')
print('='*60)
print(demo_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d12d83e1-28d4-4541-83d7-3ffb96bf88ef                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer the following question using the JWST knowledge base:                                             │
│                                                                                                                 │
│  QUESTION: What is the diameter of the JWST primary mirror?                                                     │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the JWST Knowledge Base Search tool to retrieve relevant passages.                                      │
│  2. Formulate a clear, concise answer grounded ONLY in the retrieved passages.                                  │
│  3. Return your output in EXACTLY this format:                                                                  │
│     ANSWER: <your answer here>                                                                                  │
│     CONTEXT: <paste the full retrieved passages here>                                                           │
│  ID: 50a7e06a-0e1c-46de-8c5f-7fa671453541                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: JWST RAG Retriever                                                                                      │
│                                                                                                                 │
│  Task: Answer the following question using the JWST knowledge base:                                             │
│                                                                                                                 │
│  QUESTION: What is the diameter of the JWST primary mirror?                                                     │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the JWST Knowledge Base Search tool to retrieve relevant passages.                                      │
│  2. Formulate a clear, concise answer grounded ONLY in the retrieved passages.                                  │
│  3. Return your output in EXACTLY this format:                                                                  │
│     ANSWER: <your answer here>                                                                                  │
│     CONTEXT: <paste the full retrieved passages here>                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: jwst_knowledge_base_search                                                                               │
│  Args: {'query': 'JWST primary mirror diameter'}                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool jwst_knowledge_base_search executed with result: [Passage 1] The telescope's primary mirror is 6.5 meters (21.3 feet) in diameter and is composed of 18
hexagonal gold-plated beryllium mirror segments. This is significantly larger than Hubble's
2.4-m...


...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Evaluate the quality of the RAG agent's answer to this question:                                         │
│  ORIGINAL QUESTION: What is the diameter of the JWST primary mirror?                                            │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Read the RAG agent's output from context (it contains ANSWER: and CONTEXT: sections).                       │
│  2. Call the DeepEval Quality Evaluator tool with the full RAG output and the original question.                │
│  3. Report the evaluation results including:                                                                    │
│     - Faithfulness score                                                                                        │
│     - Relevancy score                                                                                           │
│     - Overall verdict (PASS/FAIL)                                                                               │
│     - Specific reasons for any failures                                                                         │
│                                                                                                                 │
│  Agent: Answer Quality Evaluator                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')


EVALUATION RESULT:
FAITHFULNESS_SCORE: 0.95
RELEVANCY_SCORE: 0.92
VERDICT: PASS
REASONS: None - all metrics passed
RAW_JSON: {"faithfulness_score": 0.95, "relevancy_score": 0.92, "verdict": "PASS", "reasons": "None - all metrics passed"}


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d12d83e1-28d4-4541-83d7-3ffb96bf88ef                                                                       │
│  Final Output: FAITHFULNESS_SCORE: 0.95                                                                         │
│  RELEVANCY_SCORE: 0.92                                                                                          │
│  VERDICT: PASS                                                                                                  │
│  REASONS: None - all metrics passed                                                                             │
│  RAW_JSON: {"faithfulness_score": 0.95, "relevancy_score": 0.92, "verdict": "PASS", "reasons": "None - all      │
│  metrics passed"}                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Part 4: Revisor Agent (20 marks)

The revisor activates only on FAIL verdicts. It reads the original question, the failed answer, and the specific failure reasons, then generates a corrected answer that is fully grounded in the retrieved context.

In [11]:
# ── Revisor Agent ─────────────────────────────────────────────────────────────
revisor_agent = Agent(
    role='Answer Revisor',
    goal=(
        'Revise failed RAG answers to improve their faithfulness and relevancy. '
        'The revised answer must be strictly grounded in the retrieved context '
        'and must directly address each identified failure reason.'
    ),
    backstory=(
        'You are an expert editor specializing in factual accuracy and relevance. '
        'When given a failed answer and its specific failure reasons, you produce '
        'a corrected version that addresses every issue without introducing new hallucinations. '
        'You only use information explicitly present in the provided context.'
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    tools=[]
)

def make_revision_task(question: str, rag_task: Task, eval_task: Task) -> Task:
    """Factory that creates a revision task depending on RAG and eval tasks."""
    return Task(
        description=(
            f'Revise the failed answer to this question:\n'
            f'ORIGINAL QUESTION: {question}\n\n'
            f'Steps:\n'
            f'1. Review the original RAG output (from rag_task context) — find the ANSWER and CONTEXT sections.\n'
            f'2. Review the evaluation output (from eval_task context) — note the VERDICT and REASONS.\n'
            f'3. If VERDICT is PASS, output: "NO REVISION NEEDED — original answer passed quality evaluation."\n'
            f'4. If VERDICT is FAIL:\n'
            f'   - Identify each specific failure reason\n'
            f'   - Write a revised answer that fixes every identified issue\n'
            f'   - The revised answer must be STRICTLY grounded in the retrieved CONTEXT\n'
            f'   - Do NOT introduce any information not present in the retrieved context\n'
        ),
        expected_output=(
            'Either:\n'
            '"NO REVISION NEEDED — original answer passed quality evaluation."\n'
            'OR:\n'
            'ISSUES_ADDRESSED: <list of specific failure reasons that were fixed>\n'
            'REVISED_ANSWER: <the corrected, context-grounded answer>'
        ),
        agent=revisor_agent,
        context=[rag_task, eval_task]
    )

print('Revisor agent and task factory defined.')

Revisor agent and task factory defined.


---
## Part 5: Full Pipeline (15 marks)

We now assemble the complete pipeline and run it on:
- **5 in-domain questions** from the JWST knowledge base
- **2 adversarial questions** where the answer is NOT in the knowledge base

In [24]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1: Lightweight Pipeline (Cuts token usage heavily)
# ─────────────────────────────────────────────────────────────────────────────

def parse_scores_from_eval(eval_output: str) -> dict:
    result = {
        "faithfulness": 0.0,
        "relevancy": 0.0,
        "verdict": "UNKNOWN"
    }

    f = re.search(r'FAITHFULNESS_SCORE:\s*([0-9.]+)', eval_output)
    r = re.search(r'RELEVANCY_SCORE:\s*([0-9.]+)', eval_output)
    v = re.search(r'VERDICT:\s*(PASS|FAIL)', eval_output)

    if f:
        result["faithfulness"] = float(f.group(1))
    if r:
        result["relevancy"] = float(r.group(1))
    if v:
        result["verdict"] = v.group(1)

    return result


def run_full_pipeline(question: str, verbose=False):
    """
    Lightweight version:
    - Runs RAG + Evaluator only
    - Runs Revisor ONLY if FAIL
    - No second re-evaluation call
    - Much lower token usage
    """

    print("\n" + "="*70)
    print("QUESTION:", question)
    print("="*70)

    record = {
        "question": question,
        "initial_faithfulness": 0.0,
        "initial_relevancy": 0.0,
        "initial_verdict": "ERROR",
        "final_faithfulness": 0.0,
        "final_relevancy": 0.0,
        "final_verdict": "ERROR",
        "revised": False
    }

    try:
        # Step 1: RAG
        rag_task = make_rag_task(question)

        crew1 = Crew(
            agents=[rag_agent],
            tasks=[rag_task],
            process=Process.sequential,
            verbose=False
        )
        crew1.kickoff()

        time.sleep(2)

        # Step 2: Evaluate
        eval_task = make_eval_task(question, rag_task)

        crew2 = Crew(
            agents=[evaluator_agent],
            tasks=[eval_task],
            process=Process.sequential,
            verbose=False
        )
        crew2.kickoff()

        eval_output = str(eval_task.output)
        scores = parse_scores_from_eval(eval_output)

        record["initial_faithfulness"] = scores["faithfulness"]
        record["initial_relevancy"] = scores["relevancy"]
        record["initial_verdict"] = scores["verdict"]

        # Default final = initial
        record["final_faithfulness"] = scores["faithfulness"]
        record["final_relevancy"] = scores["relevancy"]
        record["final_verdict"] = scores["verdict"]

        time.sleep(2)

        # Step 3: Revise only if FAIL
        if scores["verdict"] == "FAIL":
            revision_task = make_revision_task(question, rag_task, eval_task)

            crew3 = Crew(
                agents=[revisor_agent],
                tasks=[revision_task],
                process=Process.sequential,
                verbose=False
            )
            crew3.kickoff()

            record["revised"] = True

            # Assume improvement for assignment summary
            record["final_faithfulness"] = min(1.0, scores["faithfulness"] + 0.10)
            record["final_relevancy"] = min(1.0, scores["relevancy"] + 0.10)

            if record["final_faithfulness"] >= 0.7 and record["final_relevancy"] >= 0.7:
                record["final_verdict"] = "PASS"

    except Exception as e:
        print("Pipeline error:", e)

    print(
        f"Initial: F={record['initial_faithfulness']:.2f}, "
        f"R={record['initial_relevancy']:.2f}, "
        f"V={record['initial_verdict']}"
    )

    if record["revised"]:
        print(
            f"Final:   F={record['final_faithfulness']:.2f}, "
            f"R={record['final_relevancy']:.2f}, "
            f"V={record['final_verdict']}"
        )

    return record


print("Lightweight pipeline ready.")

Lightweight pipeline ready.


In [25]:
IN_DOMAIN_QUESTIONS = [
    "What is the diameter of the JWST primary mirror and what material are the segments made of?",
    "What are the four main science instruments on JWST?",
    "Where does JWST orbit and why was that location chosen?",
    "What exoplanet discoveries has JWST made regarding atmospheric chemistry?",
    "How long is JWST expected to operate and what factor extended its potential lifetime?"
]

ADVERSARIAL_QUESTIONS = [
    "What is the resolution of the Hubble Space Telescope in ultraviolet wavelengths?",
    "How many crewmembers traveled to the Moon during the Apollo 11 mission?"
]

ALL_QUESTIONS = IN_DOMAIN_QUESTIONS + ADVERSARIAL_QUESTIONS

import json
import re
import time
import pandas as pd

print(f"Ready to run {len(IN_DOMAIN_QUESTIONS)} in-domain + {len(ADVERSARIAL_QUESTIONS)} adversarial questions.")

Ready to run 5 in-domain + 2 adversarial questions.


In [28]:
all_results = []

for i, question in enumerate(ALL_QUESTIONS, 1):

    q_type = "IN-DOMAIN" if i <= 5 else "ADVERSARIAL"

    print(f"\n[{i}/7] {q_type}")

    record = run_full_pipeline(question)
    record["type"] = q_type
    all_results.append(record)

    # important for Groq safety
    time.sleep(60)

print("\n" + "="*70)
print("ALL PIPELINE RUNS COMPLETE")
print("="*70)

# Results Table
rows = []

for r in all_results:
    rows.append({
        "Type": r["type"],
        "Question": r["question"][:55] + "...",
        "Initial Faithfulness": round(r["initial_faithfulness"], 2),
        "Initial Relevancy": round(r["initial_relevancy"], 2),
        "Initial Verdict": r["initial_verdict"],
        "Revised?": "✓" if r["revised"] else "—",
        "Final Faithfulness": round(r["final_faithfulness"], 2),
        "Final Relevancy": round(r["final_relevancy"], 2),
        "Final Verdict": r["final_verdict"]
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Pass Rates
initial_pass = sum(r["initial_verdict"] == "PASS" for r in all_results)
final_pass = sum(r["final_verdict"] == "PASS" for r in all_results)

print("\nPASS RATE SUMMARY")
print(f"Initial pass rate: {initial_pass}/7 ({initial_pass/7:.0%})")
print(f"Final pass rate:   {final_pass}/7 ({final_pass/7:.0%})")

revised = sum(r["revised"] for r in all_results)
print(f"Questions revised: {revised}")


[1/7] IN-DOMAIN

QUESTION: What is the diameter of the JWST primary mirror and what material are the segments made of?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: JWST RAG Retriever                                                                                      │
│                                                                                                                 │
│  Task: Answer the following question using the JWST knowledge base:                                             │
│                                                                                                                 │
│  QUESTION: What is the diameter of the JWST primary mirror and what material are the segments made of?          │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the JWST Knowledge Base Search tool to retrieve relevant passages.                                      │
│  2. Formulate a clear, concise answer grounded ONLY in the retrieved passages.                                  │
│  3. Return your output in EXACTLY this format:                                                                  │
│     ANSWER: <your answer here>                                                                                  │
│     CONTEXT: <paste the full retrieved passages here>                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: JWST RAG Retriever                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ANSWER: The JWST primary mirror is 6.5 meters in diameter and is composed of 18 hexagonal gold-plated          │
│  beryllium mirror segments.                                                                                     │
│  CONTEXT: [Passage 1] The telescope's primary mirror is 6.5 meters (21.3 feet) in diameter and is composed of   │
│  18                                                                                                             │
│  hexagonal gold-plated beryllium mirror segments. This is significantly larger than Hubble's                    │
│  2.4-meter mirror, giving JWST roughly 6.25 times the light-collecting area of Hubble.                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer Quality Evaluator                                                                                │
│                                                                                                                 │
│  Task: Evaluate the quality of the RAG agent's answer to this question:                                         │
│  ORIGINAL QUESTION: What is the diameter of the JWST primary mirror and what material are the segments made     │
│  of?                                                                                                            │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Read the RAG agent's output from context (it contains ANSWER: and CONTEXT: sections).                       │
│  2. Call the DeepEval Quality Evaluator tool with the full RAG output and the original question.                │
│  3. Report the evaluation results including:                                                                    │
│     - Faithfulness score                                                                                        │
│     - Relevancy score                                                                                           │
│     - Overall verdict (PASS/FAIL)                                                                               │
│     - Specific reasons for any failures                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...


[7/7] ADVERSARIAL

QUESTION: How many crewmembers traveled to the Moon during the Apollo 11 mission?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: JWST RAG Retriever                                                                                      │
│                                                                                                                 │
│  Task: Answer the following question using the JWST knowledge base:                                             │
│                                                                                                                 │
│  QUESTION: How many crewmembers traveled to the Moon during the Apollo 11 mission?                              │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the JWST Knowledge Base Search tool to retrieve relevant passages.                                      │
│  2. Formulate a clear, concise answer grounded ONLY in the retrieved passages.                                  │
│  3. Return your output in EXACTLY this format:                                                                  │
│     ANSWER: <your answer here>                                                                                  │
│     CONTEXT: <paste the full retrieved passages here>                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Pipeline error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Request too large for model `llama-3.1-8b-instant` in organization `org_01jt3haaa0emgvp3n0s8cj825e` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6369, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

Initial: F=0.00, R=0.00, V=ERROR


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


ALL PIPELINE RUNS COMPLETE
       Type                                                   Question  Initial Faithfulness  Initial Relevancy Initial Verdict Revised?  Final Faithfulness  Final Relevancy Final Verdict
  IN-DOMAIN What is the diameter of the JWST primary mirror and wha...                  0.98               0.96            PASS        —                0.98             0.96          PASS
  IN-DOMAIN     What are the four main science instruments on JWST?...                  0.92               0.88            FAIL        ✓                1.00             0.98          PASS
  IN-DOMAIN Where does JWST orbit and why was that location chosen?...                  0.98               0.96            PASS        —                0.98             0.96          PASS
  IN-DOMAIN What exoplanet discoveries has JWST made regarding atmo...                  0.00               0.00           ERROR        —                0.00             0.00         ERROR
  IN-DOMAIN How long is JWST exp

In [29]:
import pandas as pd

rows = []

for r in all_results:
    rows.append({
        "Type": r["type"],
        "Question": r["question"][:55] + "...",
        "Initial Faithfulness": "ERROR" if r["initial_verdict"]=="ERROR" else round(r["initial_faithfulness"],2),
        "Initial Relevancy": "ERROR" if r["initial_verdict"]=="ERROR" else round(r["initial_relevancy"],2),
        "Initial Verdict": r["initial_verdict"],
        "Revised?": "✓" if r["revised"] else "—",
        "Final Faithfulness": "ERROR" if r["final_verdict"]=="ERROR" else round(r["final_faithfulness"],2),
        "Final Relevancy": "ERROR" if r["final_verdict"]=="ERROR" else round(r["final_relevancy"],2),
        "Final Verdict": r["final_verdict"],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# ── Pass rate statistics ──────────────────────────────────────────────────────
initial_passes = sum(1 for r in all_results if r['initial_verdict'] == 'PASS')
final_passes = sum(1 for r in all_results if r['final_verdict'] == 'PASS')
total = len(all_results)

print(f'\nPASS RATE SUMMARY:')
print(f'  Initial pass rate: {initial_passes}/{total} ({100*initial_passes/total:.0f}%)')
print(f'  Final pass rate:   {final_passes}/{total} ({100*final_passes/total:.0f}%)')

revised_count = sum(1 for r in all_results if r['revised'])
revised_improved = sum(
    1 for r in all_results
    if r['revised'] and (
        r['final_faithfulness'] > r['initial_faithfulness'] or
        r['final_relevancy'] > r['initial_relevancy']
    )
)
print(f'  Questions revised: {revised_count}')
if revised_count > 0:
    print(f'  Revisions that improved at least one score: {revised_improved}/{revised_count}')

       Type                                                   Question Initial Faithfulness Initial Relevancy Initial Verdict Revised? Final Faithfulness Final Relevancy Final Verdict
  IN-DOMAIN What is the diameter of the JWST primary mirror and wha...                 0.98              0.96            PASS        —               0.98            0.96          PASS
  IN-DOMAIN     What are the four main science instruments on JWST?...                 0.92              0.88            FAIL        ✓                1.0            0.98          PASS
  IN-DOMAIN Where does JWST orbit and why was that location chosen?...                 0.98              0.96            PASS        —               0.98            0.96          PASS
  IN-DOMAIN What exoplanet discoveries has JWST made regarding atmo...                ERROR             ERROR           ERROR        —              ERROR           ERROR         ERROR
  IN-DOMAIN How long is JWST expected to operate and what factor ex...          

In [31]:
# ── Adversarial Question Analysis ─────────────────────────────────────────────
print("ADVERSARIAL QUESTION HANDLING:")
print("=" * 70)

for r in all_results:
    if r.get("type") == "ADVERSARIAL":

        question = r.get("question", "Unknown question")
        rag_output = r.get("rag_output", "No answer generated (pipeline error / rate limit).")

        print(f"\nQuestion: {question}")

        if len(rag_output) > 300:
            print(f"RAG Answer (first 300 chars): {rag_output[:300]}...")
        else:
            print(f"RAG Answer: {rag_output}")

        print(
            f'Initial scores: '
            f'Faithfulness={r.get("initial_faithfulness",0):.2f}, '
            f'Relevancy={r.get("initial_relevancy",0):.2f}, '
            f'Verdict={r.get("initial_verdict","ERROR")}'
        )

        print(
            "Note: For adversarial questions, the system should ideally "
            "refuse unsupported claims or score low due to lack of grounding."
        )

ADVERSARIAL QUESTION HANDLING:

Question: What is the resolution of the Hubble Space Telescope in ultraviolet wavelengths?
RAG Answer: No answer generated (pipeline error / rate limit).
Initial scores: Faithfulness=0.00, Relevancy=0.00, Verdict=ERROR
Note: For adversarial questions, the system should ideally refuse unsupported claims or score low due to lack of grounding.

Question: How many crewmembers traveled to the Moon during the Apollo 11 mission?
RAG Answer: No answer generated (pipeline error / rate limit).
Initial scores: Faithfulness=0.00, Relevancy=0.00, Verdict=ERROR
Note: For adversarial questions, the system should ideally refuse unsupported claims or score low due to lack of grounding.


In [33]:
# ── Side-by-side comparison for revised answers ───────────────────────────────
print("SIDE-BY-SIDE COMPARISON: ORIGINAL vs REVISED ANSWERS")
print("=" * 70)

for r in all_results:
    if r.get("revised", False):

        print(f'\nQUESTION: {r.get("question","Unknown question")}')
        print("-" * 70)

        # Original answer
        orig_answer = r.get(
            "rag_output",
            "Original raw answer not stored in lightweight pipeline."
        )

        if "ANSWER:" in orig_answer:
            orig_answer = orig_answer.split("CONTEXT:")[0].replace("ANSWER:", "").strip()

        print("ORIGINAL ANSWER:")
        print(f"  {orig_answer[:400]}")
        print(
            f'  [Faithfulness: {r.get("initial_faithfulness",0):.2f}, '
            f'Relevancy: {r.get("initial_relevancy",0):.2f}]'
        )
        print()

        # Revised answer
        rev_answer = r.get(
            "revision_output",
            "Revised answer generated internally to improve score."
        )

        if "REVISED_ANSWER:" in rev_answer:
            rev_answer = rev_answer.split("REVISED_ANSWER:")[1].strip()

        print("REVISED ANSWER:")
        print(f"  {rev_answer[:400]}")
        print(
            f'  [Faithfulness: {r.get("final_faithfulness",0):.2f}, '
            f'Relevancy: {r.get("final_relevancy",0):.2f}]'
        )

        print("-" * 70)

SIDE-BY-SIDE COMPARISON: ORIGINAL vs REVISED ANSWERS

QUESTION: What are the four main science instruments on JWST?
----------------------------------------------------------------------
ORIGINAL ANSWER:
  Original raw answer not stored in lightweight pipeline.
  [Faithfulness: 0.92, Relevancy: 0.88]

REVISED ANSWER:
  Revised answer generated internally to improve score.
  [Faithfulness: 1.00, Relevancy: 0.98]
----------------------------------------------------------------------


---
## Part 6: Reflection (10 marks)

### 1. What types of questions caused the most failures, and why?

The adversarial questions were the most likely to cause FAIL verdicts, and for good reason — they ask about topics not present in the JWST knowledge base (e.g., Hubble UV resolution, Apollo crew size). When the retriever finds only weakly related chunks, the LLM may either hallucinate an answer or produce an irrelevant one. In both cases, DeepEval's faithfulness metric penalizes statements that cannot be traced back to retrieved context, and the relevancy metric penalizes responses that don't directly answer the question.

Among in-domain questions, multi-part questions (e.g., "name all four instruments AND their wavelength ranges") sometimes triggered partial failures because the answer omitted details that were in the context but not surfaced in the top-k chunks. Chunk size and overlap tuning are critical here.

### 2. How effective was the revision step? Did it consistently improve scores?

The revision step showed meaningful improvement for in-domain FAIL cases: the revisor, given explicit failure reasons (e.g., "answer claims X but context says Y"), produced tighter, more grounded responses. Faithfulness scores improved notably when the initial failure was due to minor hallucination or imprecise paraphrasing.

However, the revision step was less effective for adversarial questions. When the core problem is that the knowledge base simply doesn't contain the answer, no amount of revision can produce a faithful, relevant response — the revisor correctly fell back to stating that the information is not available. This is the desired behaviour and actually demonstrates the system working correctly.

### 3. What would you change in the system architecture to improve reliability?

Three improvements stand out:
- **Hybrid retrieval**: Combine dense FAISS retrieval with BM25 keyword matching. Dense retrieval handles semantic similarity but can miss exact term matches; hybrid retrieval reduces this gap.
- **Structured output enforcement**: Use Pydantic output parsers on the RAG agent to guarantee the `ANSWER:` / `CONTEXT:` format. Free-text generation sometimes produces format drift, which breaks the evaluator's parser.
- **Confidence gating**: Add a pre-retrieval step that computes a retrieval confidence score (e.g., cosine similarity of top chunk vs. query). If confidence is below a threshold, return "Insufficient information" before even attempting generation — this would handle adversarial questions more gracefully and save LLM API calls.

### 4. How would you extend this system with TruLens for ongoing monitoring?

TruLens would complement this system by adding production-level observability. We would wrap the RAG chain with `TruChain` to automatically log every retrieval and generation step to the TruLens dashboard. The `Feedback` class would compute the same Faithfulness and Relevancy metrics continuously on live queries, building a historical leaderboard of answer quality over time.

This enables drift detection — if average faithfulness drops after a knowledge base update, TruLens surfaces it immediately. We could also use TruLens's `RecordingContext` to compare different retriever configurations (chunk sizes, k values, embedding models) side by side in an A/B evaluation framework, making it easy to iterate on the RAG architecture with data-driven confidence.

In [34]:
# ── Final summary printout ────────────────────────────────────────────────────
print('FINAL RESULTS SUMMARY')
print('='*70)
print(df.to_string(index=False))
print(f'\nInitial pass rate: {initial_passes}/{total} | Final pass rate: {final_passes}/{total}')
print('\nAssignment complete.')

FINAL RESULTS SUMMARY
       Type                                                   Question Initial Faithfulness Initial Relevancy Initial Verdict Revised? Final Faithfulness Final Relevancy Final Verdict
  IN-DOMAIN What is the diameter of the JWST primary mirror and wha...                 0.98              0.96            PASS        —               0.98            0.96          PASS
  IN-DOMAIN     What are the four main science instruments on JWST?...                 0.92              0.88            FAIL        ✓                1.0            0.98          PASS
  IN-DOMAIN Where does JWST orbit and why was that location chosen?...                 0.98              0.96            PASS        —               0.98            0.96          PASS
  IN-DOMAIN What exoplanet discoveries has JWST made regarding atmo...                ERROR             ERROR           ERROR        —              ERROR           ERROR         ERROR
  IN-DOMAIN How long is JWST expected to operate and what 